# Transformer block lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/transformer-block.ipynb)

Run a complete, tiny, modern transformer forward pass in NumPy and poke at it. Companion to [The transformer block](../04-architectures/transformer.md). Runs on CPU in seconds. The model is untrained: this lab is about how data flows, not what it has learned.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

import math
import numpy as np
from s2a.lm import cross_entropy
from s2a.transformer import (TransformerConfig, LLAMA3_8B, init_params, forward,
                             block, rms_norm, attention, swiglu, count_params)

## 1. Build a tiny model and run it

In [ ]:
cfg = TransformerConfig(vocab_size=100, d_model=64, n_layers=4, n_heads=4, n_kv_heads=2, d_ff=172)
params = init_params(cfg, np.random.default_rng(0))
ids = [5, 17, 42, 8, 99, 3]

logits = forward(params, cfg, ids)
print("tokens in:", len(ids), "-> logits out:", logits.shape, "(one row of vocabulary scores per position)")
print("parameters:", f"{count_params(params):,}", "| formula:", f"{cfg.param_count():,}")

## 2. Follow the residual stream

Each block reads the stream and **adds** to it. Step through by hand and watch how much each sublayer writes.

In [ ]:
x = params["embedding"][ids]
size = lambda a: float(np.linalg.norm(a, axis=-1).mean())
print(f"after embedding: stream size {size(x):.3f}")
for i, layer in enumerate(params["layers"]):
    a = attention(rms_norm(x, layer["norm1"]), layer, cfg)
    x = x + a
    m = swiglu(rms_norm(x, layer["norm2"]), layer["w_gate"], layer["w_up"], layer["w_down"])
    x = x + m
    print(f"block {i}: attention wrote {size(a):.3f}, MLP wrote {size(m):.3f}, stream size now {size(x):.3f}")

print("matches forward():", np.allclose(rms_norm(x, params["final_norm"]) @ params["output_head"].T, logits))

## 3. Causality: the future can't affect the past

Change the last two tokens. Outputs at earlier positions don't move at all. This is what makes parallel training and the KV cache possible.

In [ ]:
changed = forward(params, cfg, ids[:4] + [1, 2])
for pos in range(len(ids)):
    print(f"position {pos}: max change in logits {np.abs(changed[pos] - logits[pos]).max():.6f}")

## 4. An untrained model knows nothing

Its loss should sit near `ln(vocab_size)`, the uniform-guess baseline from the [language-modeling lab](lm-objective.ipynb).

In [ ]:
seq = list(np.random.default_rng(1).integers(0, cfg.vocab_size, size=60))
loss = cross_entropy(forward(params, cfg, seq[:-1]), np.array(seq[1:]))
print(f"untrained loss {loss:.3f} vs ln({cfg.vocab_size}) = {math.log(cfg.vocab_size):.3f}")

## 5. Where do a real model's parameters live?

Same formula, Llama 3 8B's published shape.

In [ ]:
total = LLAMA3_8B.param_count()
print(f"Llama 3 8B: {total:,} parameters")
for name, n in LLAMA3_8B.param_breakdown().items():
    print(f"  {name:<12} {n / 1e6:>8,.1f} M  {100 * n / total:5.1f}%  {'#' * round(40 * n / total)}")

## Try it

- Set `n_kv_heads` equal to `n_heads` (plain multi-head attention), then to 1 (multi-query). How does the parameter count change? How would the KV cache change?
- Remove the `rms_norm` calls inside the loop in section 2 and use 40 layers with larger random weights. What happens to the stream size?
- Swap two input tokens. Which positions' outputs change, and why does RoPE matter here?